In [3]:
from osgeo import gdal
import os
import pandas as pd

In [4]:
def list_folders(main_path):
    """
    Return a list of absolute paths (with forward slashes)
    of all immediate subfolders inside main_path.
    """
    folder_list = []

    for entry in os.listdir(main_path):
        full_path = os.path.join(main_path, entry)
        if os.path.isdir(full_path):
            folder_list.append(full_path.replace("\\", "/"))

    # If main_path has no subfolders, return itself
    if not folder_list:
        folder_list = [main_path]

    return folder_list


def find_folders_tifs(main_path, suffix=None):
    """
    Recursively search inside main_path and all its subfolders
    and return:
    - list of all .tif/.tiff files
    - dictionary {folder_name: [tif files]}
    """
    folder_tif_dict = {}
    tif_files = []

    for root, dirs, files in os.walk(main_path):
        folder_name = os.path.basename(root)

        # Apply suffix filter only if provided
        if suffix is None or folder_name.endswith(suffix):
            for f in files:
                if f.lower().endswith((".tif", ".tiff")):
                    full_path = os.path.join(root, f).replace("\\", "/")

                    tif_files.append(full_path)

                    # Initialize list per folder if not exists
                    if folder_name not in folder_tif_dict:
                        folder_tif_dict[folder_name] = []

                    folder_tif_dict[folder_name].append(full_path)

    return tif_files, folder_tif_dict


def quality_check(File_list):
    """
    Check for:
    - corrupted raster files
    - missing nodata values
    """
    corrupted_files = {}
    nodata_files = {}

    for file in File_list:
        dataset = gdal.Open(file)

        if dataset is None:
            corrupted_files[file] = "Unable to open raster file"
        else:
            band = dataset.GetRasterBand(1)

            if band is None:
                corrupted_files[file] = "Unable to access raster band"
            else:
                nodata_value = band.GetNoDataValue()

                if nodata_value is None:
                    nodata_files[file] = "The nodata value is not set"

    return corrupted_files, nodata_files

In [6]:
"""Inputs"""
main_path = r"Y:\z_resources\un_gbf\01_aggregation_phase\02_mask_rasters"
folder_list = list_folders(main_path)
# File_list, folder_tif_dict = find_folders_tifs(main_path)

In [8]:
folder_list[1:]

['Y:/z_resources/un_gbf/01_aggregation_phase/02_mask_rasters/global_wetlands_20m_2021_mosaic',
 'Y:/z_resources/un_gbf/01_aggregation_phase/02_mask_rasters/global_wetlands_20m_2022_mosaic']

In [ ]:
"""This is a quality check for the nodata value"""

corrupted_files, nodata_files = quality_check(File_list)
quality_check_df = pd.DataFrame({"Corrupted files": corrupted_files, "Nodata files": nodata_files})
print(quality_check_df)

In [9]:
"""
- Get a list of the raster files inside the folder.
- Do the geoprocessing per each file
"""
#beware if the folder indexation is okay
for folder in folder_list[1:]: #[:10]
    print("starting: " + folder)
    File_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
    for file in os.listdir(folder):
        if file.endswith(".tiff") or file.endswith(".tif"):
            if file not in File_list:
                File_list.append(os.path.join(folder, file).replace("\\","/"))
        else:
            pass
        
    output = os.path.join(folder, os.path.basename(folder) + "_merged_total.tif") #if the output is the name of the folder
    # for a single output
    # output = "vcs_2020_global_300m_1.tif"

    # Specify the desired pixel size in the output
    # x_resolution = 100  # Horizontal pixel size
    # y_resolution = 100  # Vertical pixel size

    """Geoprocessing"""
    merged_tif = gdal.Warp(output, File_list, format="GTiff",
            # outputType = gdal.GDT_Byte, # this thing converts nodata values to zeros
            # srcNodata = -32768, # gives a value to the nodata areas
            # srcWin = [str(180), str(84), str(-180), str(-57)]
            # xRes=x_resolution, yRes=y_resolution,
            dstNodata = 255, # sets the no data value
            # dstSRS = 'EPSG:3035',
            creationOptions=["COMPRESS=DEFLATE", "TILED=YES"])
    
    """remove the color pallete""" # optional         
    # band = merged_tif.GetRasterBand(1)
    # band.SetRasterColorTable(None)
    # # Close file and flush to disk
    # del band

    merged_tif = None 
    print(output)

starting: Y:/z_resources/un_gbf/01_aggregation_phase/02_mask_rasters/global_wetlands_20m_2021_mosaic
Y:/z_resources/un_gbf/01_aggregation_phase/02_mask_rasters/global_wetlands_20m_2021_mosaic\global_wetlands_20m_2021_mosaic_merged_total.tif
starting: Y:/z_resources/un_gbf/01_aggregation_phase/02_mask_rasters/global_wetlands_20m_2022_mosaic
Y:/z_resources/un_gbf/01_aggregation_phase/02_mask_rasters/global_wetlands_20m_2022_mosaic\global_wetlands_20m_2022_mosaic_merged_total.tif


In [ ]:
"""File parser / For testing"""

path = r"Z:\z_resources\ruben\utci_2021\raster_output"
File_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
for file in os.listdir(path): 
    if ".tif" in file:
        if file not in File_list:
            # File_list.append(os.path.join(path, file).replace("\\","/"))
            File_list.append(file)
    else:
        pass
    
    file_string = " ".join(File_list)
    print(file_string)

In [17]:
os.chdir(path)
text_file = open("sample.txt", "w")
n = text_file.write(file_string)
text_file.close()

In [6]:
print(os.getcwd())

\\akif.internal\public\z_resources\ruben\wb_temporal\cstorage_gabon2020
